<a href="https://colab.research.google.com/github/Naomy-Yailin/SIS420/blob/main/Laboratorio_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# PyTorch Regresión Lineal y Logística con Checkpoints


In [ ]:
LIBRERIAS

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import os

# Crear carpeta de checkpoints
os.makedirs('checkpoints', exist_ok=True)

GENERA DATASETS DE LOS ANTERIORES LABS

In [2]:
np.random.seed(42)

# Driver Behavior - regresión lineal múltiple
X_driver = np.random.rand(100, 3)
y_driver = X_driver @ np.array([2.0, -1.5, 3.0]) + np.random.randn(100)*0.5
df_driver = pd.DataFrame(X_driver, columns=['feature1','feature2','feature3'])
df_driver['target'] = y_driver
df_driver.to_csv('DriverBehavior.csv', index=False)

# Students Habits - regresión logística
X_students = np.random.rand(100, 4)
y_students = (X_students @ np.array([1.0, -2.0, 1.5, 0.5]) + np.random.randn(100)*0.5 > 1.5).astype(int)
df_students = pd.DataFrame(X_students, columns=['feat1','feat2','feat3','feat4'])
df_students['target'] = y_students
df_students.to_csv('StudentsHabits.csv', index=False)

In [ ]:
PREPARACION DE DATASETS CON PYTORCHS

In [3]:
# Dataset personalizado
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Cargar Driver Behavior
df_driver = pd.read_csv('DriverBehavior.csv')
dataset_driver = CustomDataset(df_driver.drop('target', axis=1).values, df_driver['target'].values)
train_size = int(0.8 * len(dataset_driver))
val_size = len(dataset_driver) - train_size
train_driver, val_driver = random_split(dataset_driver, [train_size, val_size])
train_loader_driver = DataLoader(train_driver, batch_size=16, shuffle=True)
val_loader_driver = DataLoader(val_driver, batch_size=16)

# Cargar Students Habits
df_students = pd.read_csv('StudentsHabits.csv')
dataset_students = CustomDataset(df_students.drop('target', axis=1).values, df_students['target'].values)
train_size = int(0.8 * len(dataset_students))
val_size = len(dataset_students) - train_size
train_students, val_students = random_split(dataset_students, [train_size, val_size])
train_loader_students = DataLoader(train_students, batch_size=16, shuffle=True)
val_loader_students = DataLoader(val_students, batch_size=16)

MODELOS QUE SE ESTÁ USANDO

In [4]:
# Regresión lineal múltiple
class LinearRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
    def forward(self, x):
        return self.linear(x)

input_dim_driver = df_driver.shape[1]-1
model_driver = LinearRegressionModel(input_dim_driver)

# Regresión logística
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.linear(x))

input_dim_students = df_students.shape[1]-1
model_students = LogisticRegressionModel(input_dim_students)

FUNCIÓN DE ENTRENAMIENTO Y VALIDACIÓN

In [7]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, checkpoint_path):
    best_loss = float('inf')
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)

        # Validación
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                outputs = model(X_val)
                loss = criterion(outputs, y_val)
                val_loss += loss.item() * X_val.size(0)
        val_loss /= len(val_loader.dataset)

        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}')

        # Guardar checkpoint si es el mejor
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)
            print(f'Checkpoint guardado en {checkpoint_path}')

ENTRENAMIENTO DE LOS MODELOS

In [8]:
# Driver Behavior - Regresión lineal
criterion_driver = nn.MSELoss()
optimizer_driver = optim.Adam(model_driver.parameters(), lr=0.01)
train_model(model_driver, train_loader_driver, val_loader_driver, criterion_driver, optimizer_driver, epochs=50, checkpoint_path='checkpoints/linear_driver.pth')

# Students Habits - Regresión logística
criterion_students = nn.BCELoss()
optimizer_students = optim.Adam(model_students.parameters(), lr=0.01)
train_model(model_students, train_loader_students, val_loader_students, criterion_students, optimizer_students, epochs=50, checkpoint_path='checkpoints/logistic_students.pth')

Epoch 1/50, Train Loss: 5.8506, Val Loss: 7.4874
Checkpoint guardado en checkpoints/linear_driver.pth
Epoch 2/50, Train Loss: 5.3845, Val Loss: 6.9414
Checkpoint guardado en checkpoints/linear_driver.pth
Epoch 3/50, Train Loss: 4.8999, Val Loss: 6.4419
Checkpoint guardado en checkpoints/linear_driver.pth
Epoch 4/50, Train Loss: 4.5128, Val Loss: 5.9662
Checkpoint guardado en checkpoints/linear_driver.pth
Epoch 5/50, Train Loss: 4.1083, Val Loss: 5.5386
Checkpoint guardado en checkpoints/linear_driver.pth
Epoch 6/50, Train Loss: 3.7668, Val Loss: 5.1465
Checkpoint guardado en checkpoints/linear_driver.pth
Epoch 7/50, Train Loss: 3.4813, Val Loss: 4.7835
Checkpoint guardado en checkpoints/linear_driver.pth
Epoch 8/50, Train Loss: 3.2015, Val Loss: 4.4608
Checkpoint guardado en checkpoints/linear_driver.pth
Epoch 9/50, Train Loss: 2.9434, Val Loss: 4.1806
Checkpoint guardado en checkpoints/linear_driver.pth
Epoch 10/50, Train Loss: 2.7528, Val Loss: 3.9221
Checkpoint guardado en checkpoin

CARGA DE MODELOS DESDE CHECKPOINT

In [9]:
model_driver.load_state_dict(torch.load('checkpoints/linear_driver.pth'))
model_driver.eval()

model_students.load_state_dict(torch.load('checkpoints/logistic_students.pth'))
model_students.eval()

LogisticRegressionModel(
  (linear): Linear(in_features=4, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

EVALUACIÓN

In [10]:
# Predicciones y métricas
with torch.no_grad():
    # Regresión lineal
    y_pred_driver = model_driver(torch.tensor(df_driver.drop('target', axis=1).values, dtype=torch.float32))
    mse = nn.MSELoss()(y_pred_driver, torch.tensor(df_driver['target'].values, dtype=torch.float32).view(-1,1))
    print(f'MSE Driver Behavior: {mse.item():.4f}')

    # Regresión logística
    y_pred_students = model_students(torch.tensor(df_students.drop('target', axis=1).values, dtype=torch.float32))
    y_pred_labels = (y_pred_students > 0.5).float()
    accuracy = (y_pred_labels.view(-1) == torch.tensor(df_students['target'].values, dtype=torch.float32)).float().mean()
    print(f'Accuracy Students Habits: {accuracy.item():.4f}')

MSE Driver Behavior: 1.1934
Accuracy Students Habits: 0.8900
